In [2]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

import joblib

In [3]:
ratings = pd.read_csv(
    "../data/processed/ratings_clean.csv"
)

movies = pd.read_csv(
    "../data/processed/movies_clean.csv"
)

print("Ratings:", ratings.shape)
print("Movies:", movies.shape)

Ratings: (27753444, 4)
Movies: (58098, 3)


In [4]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,307,3.5,1256677221
1,1,481,3.5,1256677456
2,1,1091,1.5,1256677471
3,1,1257,4.5,1256677460
4,1,1449,4.5,1256677264


In [5]:
print("Unique users:", ratings["userId"].nunique())
print("Unique movies rated:", ratings["movieId"].nunique())
print("Total ratings:", len(ratings))

Unique users: 283228
Unique movies rated: 53889
Total ratings: 27753444


In [6]:
print("Unique users:", ratings["userId"].nunique())
print("Unique movies rated:", ratings["movieId"].nunique())
print("Total ratings:", len(ratings))

Unique users: 283228
Unique movies rated: 53889
Total ratings: 27753444


In [8]:
movie_rating_counts = (
    ratings.groupby("movieId")
    .size()
    .sort_values(ascending=False)
)

movie_rating_counts.head(10)

movieId
318     97999
356     97040
296     92406
593     87899
2571    84545
260     81815
480     76451
527     71516
110     68803
1       68469
dtype: int64

In [9]:
popular_movie_ids = movie_rating_counts[
    movie_rating_counts >= 20
].index

print(
    "Movies with at least 20 ratings:",
    len(popular_movie_ids)
)

Movies with at least 20 ratings: 18366


In [11]:
filtered_ratings = ratings[
    ratings["movieId"].isin(popular_movie_ids)
].copy()

print("Filtered ratings:", filtered_ratings.shape)

Filtered ratings: (27587850, 4)


In [1]:
user_movie_matrix = filtered_ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating",
    fill_value=0
)

print(
    "User-Movie matrix:",
    user_movie_matrix.shape
)

NameError: name 'filtered_ratings' is not defined

In [2]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
ratings = pd.read_csv(
    "../data/processed/ratings_clean.csv"
)

movies = pd.read_csv(
    "../data/processed/movies_clean.csv"
)

print("Ratings:", ratings.shape)
print("Movies:", movies.shape)

Ratings: (27753444, 4)
Movies: (58098, 3)


In [4]:
movie_counts = ratings["movieId"].value_counts()

popular_movie_ids = movie_counts[
    movie_counts >= 20
].index

print("Movies with at least 20 ratings:",
      len(popular_movie_ids))

Movies with at least 20 ratings: 18366


In [5]:
filtered_ratings = ratings[
    ratings["movieId"].isin(popular_movie_ids)
].copy()

print("Filtered ratings:", filtered_ratings.shape)

Filtered ratings: (27587850, 4)


In [6]:
user_ids = filtered_ratings["userId"].unique()
movie_ids = filtered_ratings["movieId"].unique()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

movie_to_index = {
    movie_id: index
    for index, movie_id in enumerate(movie_ids)
}

In [8]:
print("Users:", len(user_ids))
print("Movies:", len(movie_ids))

Users: 283172
Movies: 18366


In [9]:
row_indices = filtered_ratings["userId"].map(
    user_to_index
)

col_indices = filtered_ratings["movieId"].map(
    movie_to_index
)

values = filtered_ratings["rating"].astype(np.float32)

In [10]:
user_movie_sparse = csr_matrix(
    (
        values,
        (row_indices, col_indices)
    ),
    shape=(
        len(user_ids),
        len(movie_ids)
    ),
    dtype=np.float32
)

In [11]:
user_movie_sparse = csr_matrix(
    (
        values,
        (row_indices, col_indices)
    ),
    shape=(
        len(user_ids),
        len(movie_ids)
    ),
    dtype=np.float32
)

In [12]:
index_to_movie = {
    index: movie_id
    for movie_id, index in movie_to_index.items()
}

In [13]:
def collaborative_recommend(
    movie_id,
    num_recommendations=10
):
    
    if movie_id not in movie_to_index:
        return "Movie not available in collaborative filtering data."
    
    movie_index = movie_to_index[movie_id]
    
    # Get selected movie's ratings
    movie_vector = user_movie_sparse[
        :, movie_index
    ].T
    
    # Compare this movie with all movies
    similarities = cosine_similarity(
        movie_vector,
        user_movie_sparse.T
    ).flatten()
    
    # Get highest similarity
    similar_indices = similarities.argsort()[
        -(num_recommendations + 1):
    ][::-1]
    
    # Remove the selected movie
    similar_indices = [
        i for i in similar_indices
        if i != movie_index
    ][:num_recommendations]
    
    recommended_ids = [
        index_to_movie[i]
        for i in similar_indices
    ]
    
    result = movies[
        movies["movieId"].isin(recommended_ids)
    ][
        ["movieId", "title", "genres"]
    ].copy()
    
    similarity_map = {
        index_to_movie[i]: round(
            similarities[i], 3
        )
        for i in similar_indices
    }
    
    result["similarity_score"] = (
        result["movieId"].map(similarity_map)
    )
    
    result = result.sort_values(
        "similarity_score",
        ascending=False
    )
    
    return result

In [14]:
movies.head(10)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [16]:
collaborative_recommend(1)

,movieId,title,genres,similarity_score
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Sci-Fi,0.538
767,780,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller,0.536
3028,3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.525
1242,1270,Back to the Future (1985),Adventure|Comedy|Sci-Fi,0.520
476,480,Jurassic Park (1993),Action|Adventure|Sci-Fi|Thriller,0.512
352,356,Forrest Gump (1994),Comedy|Drama|Romance|War,0.508
360,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,0.501
640,648,Mission: Impossible (1996),Action|Adventure|Mystery|Thriller,0.497
1184,1210,Star Wars: Episode VI - Return of the Jedi (1983),Action|Adventure|Sci-Fi,0.497
582,588,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,0.492


In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz
import joblib

In [2]:
ratings = pd.read_csv("../data/processed/ratings_clean.csv")
movies = pd.read_csv("../data/processed/movies_clean.csv")

print("Ratings:", ratings.shape)
print("Movies:", movies.shape)

Ratings: (27753444, 4)
Movies: (58098, 3)


In [3]:
movie_counts = ratings["movieId"].value_counts()

popular_movie_ids = movie_counts[movie_counts >= 20].index

filtered_ratings = ratings[
    ratings["movieId"].isin(popular_movie_ids)
].copy()

print("Movies with at least 20 ratings:", len(popular_movie_ids))
print("Filtered ratings:", filtered_ratings.shape)

Movies with at least 20 ratings: 18366
Filtered ratings: (27587850, 4)


In [4]:
user_ids = filtered_ratings["userId"].unique()
movie_ids = filtered_ratings["movieId"].unique()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

movie_to_index = {
    movie_id: index
    for index, movie_id in enumerate(movie_ids)
}

print("Users:", len(user_ids))
print("Movies:", len(movie_ids))

Users: 283172
Movies: 18366


In [5]:
row_indices = filtered_ratings["userId"].map(user_to_index)
col_indices = filtered_ratings["movieId"].map(movie_to_index)

values = filtered_ratings["rating"].astype(np.float32)

user_movie_sparse = csr_matrix(
    (
        values,
        (row_indices, col_indices)
    ),
    shape=(len(user_ids), len(movie_ids)),
    dtype=np.float32
)

print("Sparse matrix shape:", user_movie_sparse.shape)
print("Number of ratings:", user_movie_sparse.nnz)

Sparse matrix shape: (283172, 18366)
Number of ratings: 27587850


In [6]:
index_to_movie = {
    index: movie_id
    for movie_id, index in movie_to_index.items()
}

In [7]:
save_npz(
    "../models/user_movie_sparse.npz",
    user_movie_sparse
)

joblib.dump(
    user_to_index,
    "../models/user_to_index.pkl"
)

joblib.dump(
    movie_to_index,
    "../models/movie_to_index.pkl"
)

joblib.dump(
    index_to_movie,
    "../models/index_to_movie.pkl"
)

print("All collaborative filtering files saved successfully!")

All collaborative filtering files saved successfully!


In [8]:
import os

file_path = "../models/user_movie_sparse.npz"

print("File exists:", os.path.exists(file_path))
print("File size:", os.path.getsize(file_path), "bytes")

File exists: True
File size: 61048205 bytes


In [9]:
from scipy.sparse import load_npz

test_sparse = load_npz(
    "../models/user_movie_sparse.npz"
)

print("Loaded successfully!")
print("Shape:", test_sparse.shape)
print("Ratings:", test_sparse.nnz)

Loaded successfully!
Shape: (283172, 18366)
Ratings: 27587850
